# 🎯 Word-Level Training — LightGBM GPU
**Approach:** LightGBM with GPU mode
**Why:** Different CUDA requirements than PyTorch
**Alternative to:** PyTorch CUDA issues on Kaggle P100


In [ ]:
# Cell 1: Setup
import os, warnings
warnings.filterwarnings('ignore')

DATA_BASE = '/kaggle/input/datasets/subhajitdas/chucklenet-48v-wordlevel'
FEAT_DIR = f'{DATA_BASE}/features'
LABEL_DIR = f'{DATA_BASE}/labels'

import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
import time

print(f'LightGBM version: {lgb.__version__}')
print(f'GPU support: {"CUDA" in lgb.LGBMClassifier().get_params()}')


In [ ]:
# Cell 2: Load data
X_list, y_list, vids_list = [], [], []

for f in sorted(os.listdir(FEAT_DIR)):
    if not f.endswith('_features.npy'): continue
    vid = f.replace('_features.npy', '')
    label_path = f'{LABEL_DIR}/{vid}_labels.npy'
    if not os.path.exists(label_path): continue
    try:
        X = np.load(f'{FEAT_DIR}/{f}')
        y = np.load(label_path)
        n = min(len(X), len(y))
        X_list.append(X[:n])
        y_list.append(y[:n])
        vids_list.extend([vid] * n)
    except: continue

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
vids_all = np.array(vids_list)
X_all = np.nan_to_num(X_all.astype(np.float32), nan=0.0)

unique_vids = sorted(set(vids_list))
pos_rate = y_all.mean()

print(f'Videos: {len(unique_vids)}, Words: {len(y_all)}, Pos rate: {100*pos_rate:.1f}%')

In [ ]:
# Cell 3: Train LightGBM
t0 = time.time()

gkf = GroupKFold(n_splits=min(5, len(unique_vids)))
fold_f1s = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, vids_all)):
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]
    
    if yte.sum()==0 or ytr.sum()==0: continue
    
    # Try GPU first, fall back to CPU
    try:
        model = lgb.LGBMClassifier(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=63,
            scale_pos_weight=(1-pos_rate)/max(pos_rate, 0.01),
            device='gpu',
            gpu_platform_id=0,
            gpu_device_id=0,
            verbosity=-1,
            random_state=42
        )
        model.fit(Xtr, ytr)
        print(f'Fold {fold+1}: GPU mode')
    except Exception as e:
        print(f'Fold {fold+1}: GPU failed ({str(e)[:50]}), trying CPU...')
        model = lgb.LGBMClassifier(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=63,
            scale_pos_weight=(1-pos_rate)/max(pos_rate, 0.01),
            verbosity=-1,
            random_state=42
        )
        model.fit(Xtr, ytr)
    
    probs = model.predict_proba(Xte)[:, 1]
    p = precision_score(yte, (probs>=0.5).astype(int), zero_division=0)
    r = recall_score(yte, (probs>=0.5).astype(int), zero_division=0)
    f = f1_score(yte, (probs>=0.5).astype(int), zero_division=0)
    fold_f1s.append(f)
    print(f'Fold {fold+1}: F1={f:.4f} P={p:.4f} R={r:.4f} ({time.time()-t0:.0f}s)')

print(f'\nCV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')

In [ ]:
# Cell 4: Save
import joblib
joblib.dump(model, '/kaggle/working/lgb_model.pkl')
import json
results = {
    'n_videos': len(unique_vids),
    'n_words': int(len(y_all)),
    'positive_rate': float(pos_rate),
    'cv_f1': float(np.mean(fold_f1s)),
    'cv_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s]
}
with open('/kaggle/working/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))